# Trafilatura로 URL 본문 텍스트 + 대표 이미지 추출

`utils/aggregate.py`의 Selenium 기반 HTML 수집 로직을 재사용해서, Trafilatura로 본문 텍스트와 대표 이미지를 추출하는 예제 노트북입니다.


In [5]:
# 필요한 경우에만 실행: Trafilatura/Selenium이 설치되어 있지 않을 때 사용합니다.
# 이미 설치되어 있다면 이 셀은 건너뜁니다.
%pip install trafilatura selenium webdriver-manager


Note: you may need to restart the kernel to use updated packages.


In [6]:
import json
from typing import Optional, Tuple

import trafilatura
from utils.aggregate import _fetch_html_via_selenium


In [7]:
def extract_text_and_top_image(url: str) -> Tuple[Optional[str], Optional[str]]:
    """왜: 하나의 URL에서 본문 텍스트와 대표 이미지를 함께 추출하기 위함."""
    # 1) Selenium으로 HTML 스냅샷 확보 (Cloudflare 등 우회 포함)
    html = _fetch_html_via_selenium(url)

    # 2) Trafilatura로 JSON + 메타데이터 + 이미지까지 추출 시도
    try:
        json_str = trafilatura.extract(
            html,
            url=url,
            output_format="json",
            with_metadata=True,
            include_images=True,
            favor_precision=True,
        )
    except Exception as exc:
        raise RuntimeError(f"trafilatura 추출 중 오류가 발생했습니다: {exc}") from exc

    # 추출 실패 시: 텍스트만이라도 최대한 확보
    if not json_str:
        text_only = trafilatura.extract(html, url=url, favor_precision=True)
        return (text_only.strip() if text_only else None, None)

    # 3) JSON 파싱
    try:
        data = json.loads(json_str)
    except json.JSONDecodeError as exc:
        raise ValueError(f"trafilatura JSON 파싱 실패: {exc}") from exc

    # 4) 본문 텍스트 후보 (필드 이름은 버전/설정에 따라 다를 수 있어 안전하게 우선순위 처리)
    text_candidates = [
        data.get("text"),
        data.get("raw_text"),
        data.get("content"),
    ]
    main_text = None
    for candidate in text_candidates:
        if isinstance(candidate, str) and candidate.strip():
            main_text = candidate.strip()
            break

    # 5) 대표 이미지 후보 (메타데이터/이미지 필드를 최대한 일반적으로 탐색)
    image_candidates = []

    # 자주 쓰이는 단일 이미지 필드들
    for key in [
        "image",
        "image_url",
        "lead_image_url",
        "top_image",
        "og:image",
        "twitter:image",
    ]:
        value = data.get(key)
        if isinstance(value, str) and value:
            image_candidates.append(value)

    # 이미지 리스트 필드 처리 (예: images: [{src, alt, title}, ...])
    images_field = data.get("images")
    if isinstance(images_field, list) and images_field:
        first = images_field[0]
        if isinstance(first, str):
            image_candidates.append(first)
        elif isinstance(first, dict):
            src = first.get("src") or first.get("url")
            if isinstance(src, str) and src:
                image_candidates.append(src)

    top_image_url = next((c for c in image_candidates if c), None)

    return main_text, top_image_url


In [8]:
# 왜: 여러 URL에 대해 일괄 추출 결과를 확인하기 위함.
urls = [
    "https://toss.tech/article/payments-legacy-3",
    "https://tech.kakao.com/posts/770",
    "https://techblog.woowahan.com/22396/",
    "https://microservices.io//post/genaidevelopment/2025/09/10/allow-git-commit-considered-harmful.html",
    "https://techblog.gccompany.co.kr/%EA%B8%B0%EC%88%A0%EC%9D%84-%EA%B8%B0%ED%9A%8D%ED%95%98%EC%A7%80-%EC%95%8A%EB%8A%94-%EA%B8%B0%EC%88%A0%EA%B8%B0%ED%9A%8D%ED%8C%80-dae25aadd69b",
    "https://d2.naver.com/helloworld/3088532",
    "https://techblog.lycorp.co.jp/ko/techniques-for-improving-code-quality-23",
]

for url in urls:
    print("============================================================")
    print(f"URL: {url}")

    text, top_image = extract_text_and_top_image(url)

    print("\n=== 대표 이미지 URL ===")
    print(top_image)

    print("\n=== 본문 텍스트 (앞 500자) ===")
    if text:
        print(text[:500])
    else:
        print("본문 텍스트를 추출하지 못했습니다.")

    print("\n")

URL: https://toss.tech/article/payments-legacy-3

=== 대표 이미지 URL ===
https://static.toss.im/illusts/payments_legacy_seunghyun_jinyoung.png

=== 본문 텍스트 (앞 500자) ===
만약 여러분이 결제를 연동하는 개발자라면, 어떤 걸 해야한다고 생각하시나요?
실제로 결제창을 띄우고 PG사에 결제를 요청하려면 꽤 번거로운 작업이 필요합니다. UI 구현, 보안을 위한 인증 흐름, HTML Form과 결제 요청을 위한 비동기 처리, 그리고 다양한 예외 처리까지 고려해야 하죠. 이 과정에서 많은 개발자가 어려움을 겪습니다.
토스페이먼츠는 이러한 문제를 해결하기 위해 결제 SDK를 만들었습니다. 개발자가 보다 쉽게 결제를 연동할 수 있도록 번거로움을 줄이는 것이 목표였죠. 그렇다면 토스페이먼츠 결제 SDK를 사용하면 실제로 결제를 어떻게 구현할 수 있을까요?
// 1. SDK 로드 및 초기화const tossPayments = loadTossPayment(clientKey);const payment = tossPayment.payment({ customerKey });// 2. 결제 정보와 함께 결제 요청await payment.requestPayment('카드', {a


URL: https://tech.kakao.com/posts/770

=== 대표 이미지 URL ===
https://img1.kakaocdn.net/thumb/U896x0/?fname=https%3A%2F%2Ft1.kakaocdn.net%2Fkakao_tech%2Fimage%2F6fcf77a9019900001.png

=== 본문 텍스트 (앞 500자) ===
안녕하세요. My 구독은 톡클라우드 및 이모티콘 플러스와 같은 카카오톡 내 서비스를 정기구독하거나, 같이가치 플러스를 통해 정기 기부를 할 수 있도록 상품 정보부터 해지까지 전체 플로우를 제공하는 플랫폼입니다.
My 구독은 생긴지 